# Enrich Steam Listings From Batch

Loads Steam batch CSV, converts item base from USD to EUR using the ECB reference rate, applies saved float-fit curves (`smooth` / `segmented` / `hybrid`), adds discount-adjusted prices and spreads, then joins risk metrics.

In [5]:
from __future__ import annotations

import json
import math
import re
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display
from matplotlib import colors as mcolors

CONFIG = {
    "batch_csv": "data/scm_listings_batch.csv",
    "fit_json": "data/float_fit_rel_curves.json",
    "realtime_base_csv": "data/scm_realtime_base.csv",
    "risk_csv": "../skin_homog/screener_preprocess_risk/risk_metrics.csv",
    "ecb_daily_xml": "https://www.ecb.europa.eu/stats/eurofxref/eurofxref-daily.xml",
    "request_timeout_sec": 20,
    "ask_min": None,
    "ask_max": None,
    "risk_filters": {
        "steam_sales_n_min": 50,
        "downside_risk_max": 10.0,
        "tail_ratio_min": 0.9,
        "downside_14d_max": 0.12,
        "continuity_ratio_max": 3.5,
        "spread_hybrid_disc_max": 0.17,
    },
    "sort_by": ["spread_hybrid_disc", "spread_segmented_disc", "spread_smooth_disc", "ask"],
    "sort_ascending": [True, True, True, True],
    "output_filtered_csv": None,
}

BASE = Path('.').resolve()
if not (BASE / 'steam_scm_listings.py').is_file() and (BASE.parent / 'steam_listings' / 'steam_scm_listings.py').is_file():
    BASE = BASE.parent / 'steam_listings'

def resolve_path(value: str | None) -> Path | None:
    if value is None:
        return None
    p = Path(value)
    return p if p.is_absolute() else (BASE / p).resolve()

cfg = CONFIG
path_batch = resolve_path(cfg['batch_csv'])
path_fit = resolve_path(cfg['fit_json'])
path_base = resolve_path(cfg['realtime_base_csv'])
path_risk = resolve_path(cfg['risk_csv'])
path_out = resolve_path(cfg.get('output_filtered_csv'))

for label, p in [("batch_csv", path_batch), ("fit_json", path_fit), ("realtime_base_csv", path_base), ("risk_csv", path_risk)]:
    if p is None or not p.is_file():
        raise FileNotFoundError(f'{label}: expected file, got {p}')

print('BASE:', BASE)
print('batch:', path_batch)
print('fit:', path_fit)
print('realtime base:', path_base)
print('risk:', path_risk)


BASE: C:\Roman\skins_roundtrip_v1\steam_listings
batch: C:\Roman\skins_roundtrip_v1\steam_listings\data\scm_listings_batch.csv
fit: C:\Roman\skins_roundtrip_v1\steam_listings\data\float_fit_rel_curves.json
realtime base: C:\Roman\skins_roundtrip_v1\steam_listings\data\scm_realtime_base.csv
risk: C:\Roman\skins_roundtrip_v1\skin_homog\screener_preprocess_risk\risk_metrics.csv


In [6]:
STRUCTURAL_GAP = -1337.0

LEVEL_COLORS = {
    'excellent': '#1f7a1f',
    'very_good': '#5dbb63',
    'good': '#a9d18e',
    'ok': '#f2e6a7',
    'bad': '#e89b73',
    'awful': '#d65f5f',
}

COLOR_GRID = {
    'spread_smooth': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'spread_segmented': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'spread_hybrid': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'spread_smooth_disc': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'spread_segmented_disc': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'spread_hybrid_disc': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.0},
            {'label': 'very_good', 'lo': 0.0, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.20},
            {'label': 'bad', 'lo': 0.20, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'steam_sales_7d_n': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 20},
            {'label': 'bad', 'lo': 20, 'hi': 50},
            {'label': 'ok', 'lo': 50, 'hi': 100},
            {'label': 'good', 'lo': 100, 'hi': 200},
            {'label': 'very_good', 'lo': 200, 'hi': 400},
            {'label': 'excellent', 'lo': 400, 'hi': None},
        ],
    },
    'steam_sales_7d_iqr_risk%': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 4},
            {'label': 'very_good', 'lo': 4, 'hi': 7},
            {'label': 'good', 'lo': 7, 'hi': 10},
            {'label': 'ok', 'lo': 10, 'hi': 14},
            {'label': 'bad', 'lo': 14, 'hi': 20},
            {'label': 'awful', 'lo': 20, 'hi': None},
        ],
    },
    'steam_sales_7d_band_risk%': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 8},
            {'label': 'very_good', 'lo': 8, 'hi': 14},
            {'label': 'good', 'lo': 14, 'hi': 22},
            {'label': 'ok', 'lo': 22, 'hi': 32},
            {'label': 'bad', 'lo': 32, 'hi': 45},
            {'label': 'awful', 'lo': 45, 'hi': None},
        ],
    },
    'steam_sales_7d_downside_risk%': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 2.5},
            {'label': 'very_good', 'lo': 2.5, 'hi': 4.0},
            {'label': 'good', 'lo': 4.0, 'hi': 6.0},
            {'label': 'ok', 'lo': 6.0, 'hi': 8.0},
            {'label': 'bad', 'lo': 8.0, 'hi': 12.0},
            {'label': 'awful', 'lo': 12.0, 'hi': None},
        ],
    },
    'steam_sales_7d_tail_ratio': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 0.85},
            {'label': 'bad', 'lo': 0.85, 'hi': 0.90},
            {'label': 'ok', 'lo': 0.90, 'hi': 0.93},
            {'label': 'good', 'lo': 0.93, 'hi': 0.95},
            {'label': 'very_good', 'lo': 0.95, 'hi': 0.97},
            {'label': 'excellent', 'lo': 0.97, 'hi': None},
        ],
    },
    'steam_turnover_proxy': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 0.5},
            {'label': 'bad', 'lo': 0.5, 'hi': 1.0},
            {'label': 'ok', 'lo': 1.0, 'hi': 2.0},
            {'label': 'good', 'lo': 2.0, 'hi': 4.0},
            {'label': 'very_good', 'lo': 4.0, 'hi': 8.0},
            {'label': 'excellent', 'lo': 8.0, 'hi': None},
        ],
    },
    'steam_discount_risk_score': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.5},
            {'label': 'very_good', 'lo': 0.5, 'hi': 1.0},
            {'label': 'good', 'lo': 1.0, 'hi': 2.0},
            {'label': 'ok', 'lo': 2.0, 'hi': 3.0},
            {'label': 'bad', 'lo': 3.0, 'hi': 5.0},
            {'label': 'awful', 'lo': 5.0, 'hi': None},
        ],
    },
    'steam_daily_ret_3d': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': -0.08},
            {'label': 'bad', 'lo': -0.08, 'hi': -0.03},
            {'label': 'ok', 'lo': -0.03, 'hi': 0.00},
            {'label': 'good', 'lo': 0.00, 'hi': 0.03},
            {'label': 'very_good', 'lo': 0.03, 'hi': 0.07},
            {'label': 'excellent', 'lo': 0.07, 'hi': None},
        ],
    },
    'steam_daily_ret_7d': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': -0.12},
            {'label': 'bad', 'lo': -0.12, 'hi': -0.05},
            {'label': 'ok', 'lo': -0.05, 'hi': 0.00},
            {'label': 'good', 'lo': 0.00, 'hi': 0.04},
            {'label': 'very_good', 'lo': 0.04, 'hi': 0.10},
            {'label': 'excellent', 'lo': 0.10, 'hi': None},
        ],
    },
    'steam_daily_slope_7d': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': -0.02},
            {'label': 'bad', 'lo': -0.02, 'hi': -0.005},
            {'label': 'ok', 'lo': -0.005, 'hi': 0.005},
            {'label': 'good', 'lo': 0.005, 'hi': 0.015},
            {'label': 'very_good', 'lo': 0.015, 'hi': 0.03},
            {'label': 'excellent', 'lo': 0.03, 'hi': None},
        ],
    },
    'steam_daily_ema_gap_3_14': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': -0.05},
            {'label': 'bad', 'lo': -0.05, 'hi': -0.015},
            {'label': 'ok', 'lo': -0.015, 'hi': 0.015},
            {'label': 'good', 'lo': 0.015, 'hi': 0.04},
            {'label': 'very_good', 'lo': 0.04, 'hi': 0.08},
            {'label': 'excellent', 'lo': 0.08, 'hi': None},
        ],
    },
    'steam_daily_range_14d_pct': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.04},
            {'label': 'very_good', 'lo': 0.04, 'hi': 0.07},
            {'label': 'good', 'lo': 0.07, 'hi': 0.11},
            {'label': 'ok', 'lo': 0.11, 'hi': 0.16},
            {'label': 'bad', 'lo': 0.16, 'hi': 0.24},
            {'label': 'awful', 'lo': 0.24, 'hi': None},
        ],
    },
    'steam_daily_downside_14d_pct': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.02},
            {'label': 'very_good', 'lo': 0.02, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.08},
            {'label': 'ok', 'lo': 0.08, 'hi': 0.12},
            {'label': 'bad', 'lo': 0.12, 'hi': 0.18},
            {'label': 'awful', 'lo': 0.18, 'hi': None},
        ],
    },

    'continuity_ratio': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 1.5},
            {'label': 'very_good', 'lo': 1.5, 'hi': 1.7},
            {'label': 'good', 'lo': 1.7, 'hi': 2.0},
            {'label': 'ok', 'lo': 2.0, 'hi': 2.5},
            {'label': 'bad', 'lo': 2.5, 'hi': 3.0},
            {'label': 'awful', 'lo': 3.0, 'hi': None},
        ],
    },
    'n_fit_clean': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 10},
            {'label': 'bad', 'lo': 10, 'hi': 20},
            {'label': 'ok', 'lo': 20, 'hi': 40},
            {'label': 'good', 'lo': 40, 'hi': 80},
            {'label': 'very_good', 'lo': 80, 'hi': 150},
            {'label': 'excellent', 'lo': 150, 'hi': None},
        ],
    },
    'n_fit_raw': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 10},
            {'label': 'bad', 'lo': 10, 'hi': 20},
            {'label': 'ok', 'lo': 20, 'hi': 40},
            {'label': 'good', 'lo': 40, 'hi': 80},
            {'label': 'very_good', 'lo': 80, 'hi': 150},
            {'label': 'excellent', 'lo': 150, 'hi': None},
        ],
    },
    'fit_outlier_n': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0},
            {'label': 'very_good', 'lo': 0, 'hi': 1},
            {'label': 'good', 'lo': 1, 'hi': 2},
            {'label': 'ok', 'lo': 2, 'hi': 4},
            {'label': 'bad', 'lo': 4, 'hi': 8},
            {'label': 'awful', 'lo': 8, 'hi': None},
        ],
    },
    'fit_splits_n': {
        'bands': [
            {'label': 'excellent', 'lo': 1, 'hi': 2},
            {'label': 'very_good', 'lo': 0, 'hi': 1},
            {'label': 'good', 'lo': 2, 'hi': 3},
            {'label': 'ok', 'lo': 3, 'hi': 4},
            {'label': 'bad', 'lo': 4, 'hi': 5},
            {'label': 'awful', 'lo': None, 'hi': 0},
        ],
    },
    'scm_total_listings': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 20},
            {'label': 'bad', 'lo': 20, 'hi': 50},
            {'label': 'ok', 'lo': 50, 'hi': 100},
            {'label': 'good', 'lo': 100, 'hi': 200},
            {'label': 'very_good', 'lo': 200, 'hi': 400},
            {'label': 'excellent', 'lo': 400, 'hi': None},
        ],
    },
    'discount_sample_n': {
        'bands': [
            {'label': 'awful', 'lo': None, 'hi': 1},
            {'label': 'bad', 'lo': 1, 'hi': 3},
            {'label': 'ok', 'lo': 3, 'hi': 5},
            {'label': 'good', 'lo': 5, 'hi': 10},
            {'label': 'very_good', 'lo': 10, 'hi': 20},
            {'label': 'excellent', 'lo': 20, 'hi': None},
        ],
    },
    'avg_discount': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.00},
            {'label': 'very_good', 'lo': 0.00, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.15},
            {'label': 'bad', 'lo': 0.15, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
    'median_discount': {
        'bands': [
            {'label': 'excellent', 'lo': None, 'hi': 0.00},
            {'label': 'very_good', 'lo': 0.00, 'hi': 0.05},
            {'label': 'good', 'lo': 0.05, 'hi': 0.10},
            {'label': 'ok', 'lo': 0.10, 'hi': 0.15},
            {'label': 'bad', 'lo': 0.15, 'hi': 0.25},
            {'label': 'awful', 'lo': 0.25, 'hi': None},
        ],
    },
}

RISK_DISPLAY_ORDER = [
    'discount_sample_n', 'avg_discount', 'median_discount',
    'steam_sales_7d_mean', 'steam_sales_7d_median', 'steam_sales_7d_p10', 'steam_sales_7d_p25', 'steam_sales_7d_p75', 'steam_sales_7d_p90',
    'steam_sales_7d_min', 'steam_sales_7d_max', 'steam_sales_7d_n',
    'steam_sales_7d_iqr_risk%', 'steam_sales_7d_band_risk%', 'steam_sales_7d_downside_risk%', 'steam_sales_7d_tail_ratio',
    'steam_turnover_proxy', 'steam_discount_risk_score',
    'steam_daily_ret_3d', 'steam_daily_ret_7d', 'steam_daily_slope_7d', 'steam_daily_ema_gap_3_14',
    'steam_daily_range_14d_pct', 'steam_daily_downside_14d_pct',
    'trade_days', 'base_price', 'currency', 'steam_trade_currency',
    'status', 'error', 'sort_by', 'collected_at_utc', 'risk_collected_at_utc', 'key_trace', 'cap_hit', 'n_listings',
]

FORMATTERS = {
    'ask': '{:.2f}',
    'float_value': '{:.6f}',
    'paint_seed': '{:.0f}',
    'base_eur': '{:.2f}',
    'avg_discount': '{:.2%}',
    'median_discount': '{:.2%}',
    'pred_smooth_eur': '{:.2f}',
    'pred_segmented_eur': '{:.2f}',
    'pred_hybrid_eur': '{:.2f}',
    'pred_smooth_eur_disc': '{:.2f}',
    'pred_segmented_eur_disc': '{:.2f}',
    'pred_hybrid_eur_disc': '{:.2f}',
    'spread_smooth': '{:.2%}',
    'spread_segmented': '{:.2%}',
    'spread_hybrid': '{:.2%}',
    'spread_smooth_disc': '{:.2%}',
    'spread_segmented_disc': '{:.2%}',
    'spread_hybrid_disc': '{:.2%}',
    'steam_turnover_proxy': '{:.2f}',
    'scm_total_listings': '{:.0f}',
    'steam_sales_7d_mean': '{:.2f}',
    'steam_sales_7d_median': '{:.2f}',
    'steam_sales_7d_p10': '{:.2f}',
    'steam_sales_7d_p25': '{:.2f}',
    'steam_sales_7d_p75': '{:.2f}',
    'steam_sales_7d_p90': '{:.2f}',
    'steam_sales_7d_min': '{:.2f}',
    'steam_sales_7d_max': '{:.2f}',
    'steam_sales_7d_n': '{:.0f}',
    'steam_sales_7d_iqr_risk%': '{:.2f}',
    'steam_sales_7d_band_risk%': '{:.2f}',
    'steam_sales_7d_downside_risk%': '{:.2f}',
    'steam_sales_7d_tail_ratio': '{:.4f}',
    'steam_discount_risk_score': '{:.2f}',
    'steam_daily_ret_3d': '{:.2%}',
    'steam_daily_ret_7d': '{:.2%}',
    'steam_daily_slope_7d': '{:.4f}',
    'steam_daily_ema_gap_3_14': '{:.2%}',
    'steam_daily_range_14d_pct': '{:.2%}',
    'steam_daily_downside_14d_pct': '{:.2%}',
    'continuity_ratio': '{:.2f}',
    'n_fit_clean': '{:.0f}',
    'n_fit_raw': '{:.0f}',
    'fit_outlier_n': '{:.0f}',
    'fit_splits_n': '{:.0f}',
    'discount_sample_n': '{:.0f}',
    'trade_days': '{:.0f}',
    'base_price': '{:.2f}',
    'n_listings': '{:.0f}',
}

def _two_dec(s: pd.Series) -> pd.Series:
    def _one(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return np.nan
        return float(f'{float(v):.2f}')
    return s.map(_one)

def load_realtime_base(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    keep_cols = ['item', 'base_usd', 'base_eur', 'predicted_usd', 'predicted_eur', 'fx_usd_to_eur', 'fx_source', 'base_collected_at_utc', 'status', 'error']
    cols = [c for c in keep_cols if c in df.columns]
    out = df[cols].copy()
    for col in ['base_usd', 'base_eur', 'predicted_usd', 'predicted_eur', 'fx_usd_to_eur']:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out

def interp_curve(item_fit: dict[str, object] | None, float_values: pd.Series, model_name: str) -> np.ndarray:
    if not isinstance(item_fit, dict):
        return np.full(len(float_values), np.nan, dtype=float)
    try:
        x_grid = np.asarray(item_fit['x_grid'], dtype=float)
        y_grid = np.asarray(item_fit[model_name], dtype=float)
    except Exception:
        return np.full(len(float_values), np.nan, dtype=float)
    xq = pd.to_numeric(float_values, errors='coerce').to_numpy(dtype=float)
    out = np.full(len(xq), np.nan, dtype=float)
    ok = np.isfinite(xq)
    if len(x_grid) >= 2 and np.isfinite(x_grid).all() and np.isfinite(y_grid).all():
        out[ok] = np.interp(xq[ok], x_grid, y_grid, left=np.nan, right=np.nan)
    return out

def contains_any_mask(s: pd.Series, parts: list[str]) -> pd.Series:
    if not parts:
        return pd.Series(True, index=s.index)
    m = pd.Series(False, index=s.index)
    for part in parts:
        m |= s.str.contains(re.escape(part), regex=True, case=False, na=False)
    return m

def build_risk_masks(frame: pd.DataFrame, risk_filters: dict[str, float]) -> dict[str, pd.Series]:
    specs = [
        (f"steam_sales_7d_n >= {risk_filters['steam_sales_n_min']}", frame['steam_sales_7d_n'] >= risk_filters['steam_sales_n_min']),
        (f"steam_sales_7d_downside_risk% <= {risk_filters['downside_risk_max']}", frame['steam_sales_7d_downside_risk%'] <= risk_filters['downside_risk_max']),
        (f"steam_sales_7d_tail_ratio >= {risk_filters['tail_ratio_min']}", frame['steam_sales_7d_tail_ratio'] >= risk_filters['tail_ratio_min']),
        (f"steam_daily_downside_14d_pct <= {risk_filters['downside_14d_max']}", frame['steam_daily_downside_14d_pct'] <= risk_filters['downside_14d_max']),
        (f"continuity_ratio <= {risk_filters['continuity_ratio_max']}", frame['continuity_ratio'] <= risk_filters['continuity_ratio_max']),
        (f"spread_hybrid_disc <= {risk_filters['spread_hybrid_disc_max']}", frame['spread_hybrid_disc'] <= risk_filters['spread_hybrid_disc_max']),
    ]
    return {label: mask.fillna(False).astype(bool) for label, mask in specs}

def apply_risk_flags(frame: pd.DataFrame, risk_filters: dict[str, float]) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = frame.copy()
    risk_masks = build_risk_masks(out, risk_filters)
    risk_pass = pd.Series(True, index=out.index)
    for mask in risk_masks.values():
        risk_pass &= mask
    fail_matrix = pd.DataFrame({label: ~mask for label, mask in risk_masks.items()}, index=out.index)
    out['risk_pass'] = risk_pass
    out['risk_fail_count'] = fail_matrix.sum(axis=1)
    out['risk_fail_reasons'] = fail_matrix.apply(
        lambda row: ', '.join([col for col, failed in row.items() if failed]) if row.any() else '-',
        axis=1,
    )
    risk_report = pd.DataFrame(
        [
            {'rule': label, 'passed_rows': int(mask.sum()), 'failed_rows': int((~mask).sum())}
            for label, mask in risk_masks.items()
        ]
    )
    return out, risk_report

def _in_band(value: float, lo, hi) -> bool:
    if value is None or not np.isfinite(value):
        return False
    left_ok = True if lo is None else float(value) >= float(lo)
    right_ok = True if hi is None else float(value) < float(hi)
    return left_ok and right_ok

def _band_color(value: float, spec: dict[str, object]) -> str | None:
    for band in spec.get('bands', []):
        if _in_band(value, band.get('lo'), band.get('hi')):
            return LEVEL_COLORS.get(str(band.get('label')), '')
    return None

def make_fixed_grid_styler(frame: pd.DataFrame):
    def style_series(s: pd.Series):
        spec = COLOR_GRID.get(s.name)
        if spec is None or not pd.api.types.is_numeric_dtype(s):
            return [''] * len(s)
        out = []
        for v in pd.to_numeric(s, errors='coerce'):
            color = _band_color(v, spec)
            if not color:
                out.append('')
                continue
            out.append(f'background-color: {color}; color: black;')
        return out
    formatters = {k: v for k, v in FORMATTERS.items() if k in frame.columns}
    return frame.style.format(formatters, na_rep='-').apply(style_series, axis=0)


In [7]:
batch_df = pd.read_csv(path_batch)
batch_df = batch_df.rename(columns={'market_hash_name': 'item'})
batch_df['ask'] = pd.to_numeric(batch_df['ask'], errors='coerce')
batch_df['float_value'] = pd.to_numeric(batch_df['float_value'], errors='coerce')
batch_df['paint_seed'] = pd.to_numeric(batch_df['paint_seed'], errors='coerce').astype('Int64')
batch_df['scm_total_listings'] = pd.to_numeric(batch_df['scm_total_listings'], errors='coerce')

with open(path_fit, encoding='utf-8') as f:
    fit_payload = json.load(f)
fit_per_skin = fit_payload.get('per_skin', {})

risk_df = pd.read_csv(path_risk)

item_base_df = load_realtime_base(path_base)




df = batch_df.merge(item_base_df, on='item', how='left')
df = df.merge(risk_df, on='item', how='left', suffixes=('', '_risk'))




grouped = []
for item, g in df.groupby('item', sort=False):
    item_fit = fit_per_skin.get(item)
    gg = g.copy()
    for model_name in ['smooth', 'segmented', 'hybrid']:
        gg[f'rel_{model_name}'] = interp_curve(item_fit, gg['float_value'], model_name)
        gg[f'pred_{model_name}_eur'] = gg['base_eur'] * (1.0 + gg[f'rel_{model_name}'])
        gg[f'pred_{model_name}_eur_disc'] = gg[f'pred_{model_name}_eur'] * (1.0 - gg['avg_discount'])
        gg[f'spread_{model_name}'] = 1.0 - gg[f'pred_{model_name}_eur'] / gg['ask']
        gg[f'spread_{model_name}_disc'] = 1.0 - gg[f'pred_{model_name}_eur_disc'] / gg['ask']
    if isinstance(item_fit, dict):
        gg['continuity_ratio'] = item_fit.get('continuity_ratio')
        gg['n_fit_clean'] = item_fit.get('n_clean')
        gg['n_fit_raw'] = item_fit.get('n_raw')
        gg['fit_outlier_n'] = item_fit.get('outlier_n')
        splits = item_fit.get('splits', [])
        gg['fit_splits_n'] = len(splits) if isinstance(splits, list) else np.nan
    else:
        gg['continuity_ratio'] = np.nan
        gg['n_fit_clean'] = np.nan
        gg['n_fit_raw'] = np.nan
        gg['fit_outlier_n'] = np.nan
        gg['fit_splits_n'] = np.nan
    grouped.append(gg)

df = pd.concat(grouped, ignore_index=True)

money_cols = [c for c in df.columns if c.startswith('pred_') and (c.endswith('_eur') or c.endswith('_eur_disc'))]
spread_cols = [c for c in df.columns if c.startswith('spread_')]
for col in ['ask', 'ask_seller_net', 'base_usd', 'base_eur', 'avg_discount', 'median_discount', 'avg_ask', 'avg_predicted', 'base_price'] + money_cols + spread_cols:
    if col in df.columns:
        num = pd.to_numeric(df[col], errors='coerce')
        if col in spread_cols:
            df[col] = num
        elif col in ['avg_discount', 'median_discount']:
            df[col] = num
        else:
            df[col] = _two_dec(num)

show_cols = [
    'item', 'ask', 'float_value', 'paint_seed', 'base_eur', 'avg_discount',
    'pred_smooth_eur', 'pred_segmented_eur', 'pred_hybrid_eur',
    'pred_smooth_eur_disc', 'pred_segmented_eur_disc', 'pred_hybrid_eur_disc',
    'spread_smooth', 'spread_segmented', 'spread_hybrid',
    'spread_smooth_disc', 'spread_segmented_disc', 'spread_hybrid_disc',
    'steam_turnover_proxy', 'scm_total_listings'
]
risk_show_cols = [c for c in risk_df.columns if c != 'item']
display(df[[c for c in show_cols + risk_show_cols if c in df.columns]].head(5))
print('rows:', len(df))
print('base snapshot rows:', len(item_base_df))


,item,ask,float_value,paint_seed,base_eur,avg_discount,pred_smooth_eur,pred_segmented_eur,pred_hybrid_eur,pred_smooth_eur_disc,...,steam_discount_risk_score,steam_daily_ret_3d,steam_daily_ret_7d,steam_daily_slope_7d,steam_daily_ema_gap_3_14,steam_daily_range_14d_pct,steam_daily_downside_14d_pct,trade_days,steam_trade_currency,risk_collected_at_utc
0,MP9 | Airlock (Field-Tested),13.5,0.276943,194,9.74,0.00576,9.77,9.77,9.77,9.71,...,-0.038401,0.035456,0.013081,-0.005712,-0.007356,0.078899,0.072692,7,3,2026-04-26T14:57:14.883851+00:00
1,MP9 | Airlock (Field-Tested),14.0,0.301152,471,9.74,0.00576,9.67,9.66,9.66,9.61,...,-0.038401,0.035456,0.013081,-0.005712,-0.007356,0.078899,0.072692,7,3,2026-04-26T14:57:14.883851+00:00
2,MP9 | Airlock (Field-Tested),14.0,0.373600,322,9.74,0.00576,9.60,9.60,9.60,9.54,...,-0.038401,0.035456,0.013081,-0.005712,-0.007356,0.078899,0.072692,7,3,2026-04-26T14:57:14.883851+00:00
3,MP9 | Airlock (Field-Tested),14.0,0.320623,58,9.74,0.00576,9.64,9.64,9.64,9.59,...,-0.038401,0.035456,0.013081,-0.005712,-0.007356,0.078899,0.072692,7,3,2026-04-26T14:57:14.883851+00:00
4,MP9 | Airlock (Field-Tested),14.0,0.275079,98,9.74,0.00576,9.78,9.78,9.78,9.72,...,-0.038401,0.035456,0.013081,-0.005712,-0.007356,0.078899,0.072692,7,3,2026-04-26T14:57:14.883851+00:00


rows: 17498
base snapshot rows: 135


In [8]:
df_filtered = df.copy()
df_filtered, risk_report = apply_risk_flags(df_filtered, cfg['risk_filters'])
df_filtered = df_filtered.loc[df_filtered['risk_pass']].copy()

sort_by = [c for c in (cfg.get('sort_by') or []) if c in df_filtered.columns]
sort_ascending = list(cfg.get('sort_ascending') or [])
if sort_by:
    if len(sort_ascending) < len(sort_by):
        sort_ascending = sort_ascending + [True] * (len(sort_by) - len(sort_ascending))
    df_filtered = df_filtered.sort_values(sort_by, ascending=sort_ascending[:len(sort_by)], na_position='last').reset_index(drop=True)

print('full rows:', len(df), '| passed hard risk filters:', len(df_filtered))
display(risk_report)

base_display_cols = [
    'item', 'ask', 'float_value', 'paint_seed', 'base_eur', 'avg_discount',
    'pred_smooth_eur', 'pred_segmented_eur', 'pred_hybrid_eur',
    'pred_smooth_eur_disc', 'pred_segmented_eur_disc', 'pred_hybrid_eur_disc',
    'spread_smooth', 'spread_segmented', 'spread_hybrid',
    'spread_smooth_disc', 'spread_segmented_disc', 'spread_hybrid_disc',
    'continuity_ratio', 'n_fit_clean', 'n_fit_raw', 'fit_outlier_n', 'fit_splits_n',
    'steam_turnover_proxy', 'scm_total_listings', 'risk_fail_reasons'
]
risk_display_cols = [c for c in RISK_DISPLAY_ORDER if c in df_filtered.columns and c not in base_display_cols]
other_risk_cols = [
    c for c in risk_df.columns
    if c != 'item' and c not in base_display_cols and c not in risk_display_cols
]

DROP_DISPLAY_COLS = [
    'risk_fail_reasons',
    'steam_sales_7d_mean',
    'steam_sales_7d_median',
    'steam_sales_7d_p10',
    'steam_sales_7d_p25',
    'steam_sales_7d_p75',
    'steam_sales_7d_p90',
    'steam_sales_7d_min',
    'steam_sales_7d_max',
    'trade_days',
    'base_price',
    'currency',
    'steam_trade_currency',
    'error',
    'status',
    'sort_by',
    'key_trace',
    'cap_hit',
    "n_fit_raw",
]

display_cols = [
    c for c in base_display_cols + risk_display_cols + other_risk_cols
    if c in df_filtered.columns and c not in DROP_DISPLAY_COLS
]

# ITEM_ALLOW_ANY = ["Field", "Well", "Minimal"]
# ITEM_ALLOW_ANY = ["Fuel Injector"]
ITEM_ALLOW_ANY = []
ITEM_EXCLUDE_ANY = ["Fade", "Case", "Heat Treated"]

if ITEM_ALLOW_ANY:
    mask_allow = contains_any_mask(df_filtered["item"], ITEM_ALLOW_ANY)
    df_filtered = df_filtered.loc[mask_allow].copy()

if ITEM_EXCLUDE_ANY:
    mask_exclude = ~contains_any_mask(df_filtered["item"], ITEM_EXCLUDE_ANY)
    df_filtered = df_filtered.loc[mask_exclude].copy()


# display(make_fixed_grid_styler(df_filtered[display_cols].head(10)))

from IPython.display import HTML

styler = make_fixed_grid_styler(df_filtered[display_cols].head(100)).hide(axis="index")
html = styler.to_html()

display(HTML(f"""
<div style="overflow:auto; max-width:100%; max-height:700px; border:1px solid #ccc; padding:4px;">
<style>
table {{ border-collapse: separate; border-spacing: 0; }}

thead th {{
    position: sticky;
    top: 0;
    background: black;
    z-index: 3;
}}

th:first-child, td:first-child {{
    position: sticky;
    left: 0;
    background: black;
    z-index: 2;
}}

thead th:first-child {{
    z-index: 4;
}}
</style>
{html}
</div>
"""))




if path_out is not None:
    path_out.parent.mkdir(parents=True, exist_ok=True)
    df_filtered.to_csv(path_out, index=False)
    print('saved filtered csv:', path_out)


full rows: 17498 | passed hard risk filters: 8


,rule,passed_rows,failed_rows
0,steam_sales_7d_n >= 50,16447,1051
1,steam_sales_7d_downside_risk% <= 10.0,16724,774
2,steam_sales_7d_tail_ratio >= 0.9,16724,774
3,steam_daily_downside_14d_pct <= 0.12,17192,306
4,continuity_ratio <= 3.5,16414,1084
5,spread_hybrid_disc <= 0.17,11,17487


item,ask,float_value,paint_seed,base_eur,avg_discount,pred_smooth_eur,pred_segmented_eur,pred_hybrid_eur,pred_smooth_eur_disc,pred_segmented_eur_disc,pred_hybrid_eur_disc,spread_smooth,spread_segmented,spread_hybrid,spread_smooth_disc,spread_segmented_disc,spread_hybrid_disc,continuity_ratio,n_fit_clean,fit_outlier_n,fit_splits_n,steam_turnover_proxy,scm_total_listings,discount_sample_n,median_discount,steam_sales_7d_n,steam_sales_7d_iqr_risk%,steam_sales_7d_band_risk%,steam_sales_7d_downside_risk%,steam_sales_7d_tail_ratio,steam_discount_risk_score,steam_daily_ret_3d,steam_daily_ret_7d,steam_daily_slope_7d,steam_daily_ema_gap_3_14,steam_daily_range_14d_pct,steam_daily_downside_14d_pct,collected_at_utc,risk_collected_at_utc,n_listings,avg_ask,avg_predicted
Glock-18 | Nuclear Garden (Factory New),42.25,0.004452,740,25.84,-2.59%,40.42,39.81,39.99,41.47,40.84,41.03,4.32%,5.79%,5.35%,1.85%,3.35%,2.90%,3.04,203,6,2,2.98,101,8,-3.10%,149,6.96,12.06,3.90,0.9603,-0.05,10.42%,-2.48%,-0.0026,-0.90%,13.88%,9.72%,2026-04-17T19:14:15.233897+00:00,2026-04-26T16:03:08.397808+00:00,50,38.760000,37.800000
Glock-18 | Nuclear Garden (Factory New),50.22,0.000809,933,25.84,-2.59%,42.48,42.76,42.68,43.58,43.87,43.78,15.42%,14.85%,15.02%,13.23%,12.64%,12.82%,3.04,203,6,2,2.98,101,8,-3.10%,149,6.96,12.06,3.90,0.9603,-0.05,10.42%,-2.48%,-0.0026,-0.90%,13.88%,9.72%,2026-04-17T19:14:15.233897+00:00,2026-04-26T16:03:08.397808+00:00,50,38.760000,37.800000
SSG 08 | Big Iron (Minimal Wear),21.00,0.077905,702,13.17,-4.41%,17.05,17.19,17.15,17.80,17.95,17.91,18.81%,18.13%,18.34%,15.23%,14.52%,14.73%,2.27,77,2,2,1.46,55,8,-4.76%,73,4.66,10.41,4.88,0.9510,-0.09,8.99%,8.63%,0.0130,2.00%,11.28%,8.90%,2026-04-18T18:07:09.796038+00:00,2026-04-26T16:27:50.129474+00:00,50,17.060000,16.330000
